# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml huggingface_hub
from huggingface_hub import notebook_login
import unsloth
from unsloth import unsloth_train
notebook_login()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import datasets

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [5]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen3.5-2B",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 8192,
    "lora_r": 64,
    "lora_alpha": 64,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.02,
    "output_dir": "/content/drive/MyDrive/qwen2b_64x64_clean_drop_LR2_8192_weight_decay_0.01_last",
}

CONFIG

{'model_name': 'unsloth/Qwen3.5-2B',
 'max_seq_length': 8192,
 'lora_r': 64,
 'lora_alpha': 64,
 'learning_rate': 0.0002,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 8,
 'gradient_accumulation_steps': 2,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 10,
 'eval_steps': 100,
 'save_steps': 200,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': '/content/drive/MyDrive/qwen2b_128x128_clean_drop_LR1_8192_weight_decay_0.2_last'}

In [6]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
DATASET_CATALOG = {
    "OmniSVG/MMSVG-Icon": {
        "split": "train",
        "prompt_fields": ["description", "keywords", "detail", "prompt", "text"],
        "svg_fields": ["svg", "picosvg", "completion", "target"],
    },
    "xingxm/SVGX-Core-250k": {
        "split": "train",
        "prompt_fields": ["qwen_caption", "blip_caption", "name", "img_analysis", "prompt"],
        "svg_fields": ["svg", "completion", "target"],
    },
    "xingxm/SVGX-SFT-1M": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input", "query"],
        "svg_fields": ["completion", "output", "svg", "response"],
    },
    "thesantatitan/deepseek-svg-dataset": {
        "split": "train",
        "prompt_fields": ["prompt", "instruction", "input"],
        "svg_fields": ["completion", "output", "svg"],
    },
    "local_train_csv": {
        "path": "/content/drive/MyDrive/train.csv",
        "prompt_fields": ["prompt"],
        "svg_fields": ["svg"],
    }
}

# For a first run, keep to 1-2 sources.
ACTIVE_SOURCES = [
    "local_train_csv"
    # "xingxm/SVGX-SFT-1M"
    # "OmniSVG/MMSVG-Icon"
    #"thesantatitan/deepseek-svg-dataset",
    #"xingxm/SVGX-Core-250k"
]

In [7]:
# Global counter to limit debug prints
DEBUG_PRINT_LIMIT = 2
debug_prints = 0

def round_svg_coordinates(svg_text, precision=2):
    pattern = re.compile(r"[-+]?\d*\.\d+")
    def replacer(match):
        val = float(match.group(0))
        formatted = f"{val:.{precision}f}".rstrip('0').rstrip('.')
        return formatted if formatted else "0"
    return pattern.sub(replacer, svg_text)

def print_debug_diff(original_svg, rounded_svg):
    global debug_prints
    if debug_prints < DEBUG_PRINT_LIMIT and original_svg != rounded_svg:
        print("\n" + "="*40)
        print("🔍 SVG PRECISION REDUCTION CHECK")
        print("="*40)
        print(f"BEFORE (first 250 chars):\n{original_svg[:250]}...\n")
        print(f"AFTER  (first 250 chars):\n{rounded_svg[:250]}...\n")
        print(f"Length saved: {len(original_svg) - len(rounded_svg)} characters")
        print("="*40 + "\n")
        debug_prints += 1

def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)
    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}
    rounded_svg = round_svg_coordinates(svg, precision=2)
    print_debug_diff(svg, rounded_svg)
    return {"prompt": prompt, "svg": rounded_svg}


def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")
    if dataset_id == "xingxm/SVGX-SFT-1M":
      ds = load_dataset(
        "json",
        data_files=f"hf://datasets/{dataset_id}/SVGX_SFT_GEN_51k.json",
        split="train",
      )
    elif dataset_id == "local_train_csv":
        df = pd.read_csv(cfg["path"])
        ds = datasets.Dataset.from_pandas(df) # Convert pandas DataFrame to Hugging Face Dataset
    else:
      ds = load_dataset(dataset_id, split=cfg["split"])
    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))

    print(f"{dataset_id}: {len(ds)} usable rows")
    # MAX_SVG_LENGTH = 3072
    # ds = ds.filter(
    #     lambda x: len(x["svg"]) <= MAX_SVG_LENGTH,
    #     desc=f"filtering length > {MAX_SVG_LENGTH}"
    # )

    print(f"{dataset_id}: {len(ds)} usable rows after length filtering")
    return ds


In [8]:
datasets_ok = []
for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Loading local_train_csv ...


normalizing local_train_csv:   0%|          | 0/50000 [00:00<?, ? examples/s]


🔍 SVG PRECISION REDUCTION CHECK
BEFORE (first 250 chars):
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#FF6A00" fill-opacity="1.0"  filling="0" d="M93.30000305175781 21.20000457763672 L93.30000305175781 80.4000015258789 L21.20000457763672...

AFTER  (first 250 chars):
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#FF6A00" fill-opacity="1"  filling="0" d="M93.3 21.2 L93.3 80.4 L21.2 80.4 L21.2 179.6 L120.4 179.6 L120.4 107.1 L179.1 107.1 L179.1 21.2 L93.3 21....

Length saved: 2606 characters


🔍 SVG PRECISION REDUCTION CHECK
BEFORE (first 250 chars):
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#040000" fill-opacity="1.0"  filling="0" d="M100.3030014038086 11.76300048828125 C51.808998107910156 11.76300048828125 12.4940032958984...

AFTER  (first 250 chars):
<svg xmlns="http://www.w3.org/2000

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

local_train_csv: 50000 usable rows
local_train_csv: 50000 usable rows after length filtering
Train rows: 49000
Eval rows: 1000


{'prompt': 'A black bathtub icon with a faucet, isolated on a white background.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="" fill-opacity="1"  filling="0" d="M48.67 92.66 L48.67 41.41 L60.27 41.41 L72.81 53.91 L83.16 43.55 L66.37 26.76 L34.02 26.76 L34.02 92.66 L26.84 92.66 L45.12 165.9 L56.02 165.9 L56.02 173.24 L70.66 173.24 L70.66 165.9 L129.22 165.9 L129.22 173.24 L143.87 173.24 L143.87 165.9 L154.88 165.9 L173.16 92.66 L48.67 92.66 Z"></path></svg>'}

In [9]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map:   0%|          | 0/49000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
A black bathtub icon with a faucet, isolated on a white background.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="" fill-opacity="1"  filling="0" d="M4


In [10]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=False,
)


==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [12]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

Total parameters: 2,256,888,640
Trainable parameters: 43,646,976
Trainable %: 1.9339%


In [13]:
from transformers import TrainingArguments, IntervalStrategy
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy=IntervalStrategy.STEPS,
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
)

train_result = unsloth_train(trainer)
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/49000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,000 | Num Epochs = 1 | Total steps = 3,063
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 43,646,976 of 2,256,888,640 (1.93% trained)


Step,Training Loss,Validation Loss
100,0.519551,0.527853
200,0.535526,0.511383
300,0.521349,0.500927
400,0.505309,0.495972
500,0.507014,0.487471
600,0.507427,0.482982
700,0.489540,0.480019
800,0.479216,0.474090
900,0.483615,0.469487
1000,0.460678,0.467312


TrainOutput(global_step=3063, training_loss=0.4679993603924618, metrics={'train_runtime': 6948.0181, 'train_samples_per_second': 7.052, 'train_steps_per_second': 0.441, 'total_flos': 5.2522526065469645e+17, 'train_loss': 0.4679993603924618, 'epoch': 1.0})

## Kaggle Submission Inference Setup

**Instructions for Kaggle:**
1. Download the `qwen2b_svg_lora` folder generated by this training notebook.
2. Go to Kaggle, click **Create -> New Dataset**, and upload that folder. Name it something like `my-qwen2b-svg-lora`.
3. Create a **New Notebook** on Kaggle for your submission.
4. Click **Add Input** on the right sidebar and attach your new dataset.
5. Copy and paste the code below into your Kaggle submission notebook. Make sure to update `LORA_WEIGHTS_PATH` to match your dataset's path!

In [14]:
os.makedirs("CONFIG[output_dir]", exist_ok=True)
trainer.save_model("CONFIG[output_dir]")
tokenizer.save_pretrained("CONFIG[output_dir]")

print("Saved adapter + tokenizer to: CONFIG[output_dir]")

Saved adapter + tokenizer to: CONFIG[output_dir]


In [28]:
FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=1024):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(text=[chat_text], return_tensors="pt").to(model.device)
    # Measure how many tokens were in your prompt
    input_length = inputs.input_ids.shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    decoded_new = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True)
    # decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded_new)

    # if not is_valid_svg(svg):
    #     print("failed")
    #     svg = fallback_svg(prompt)

    return svg


#test_prompt = "a simple blue bird icon"
#test_prompt = "The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background."
test_prompt = "Give me a red square"
pred_svg = generate_svg(test_prompt)
print(pred_svg)
print("Valid SVG:", is_valid_svg(pred_svg))

<svg height="128" viewBox="0 0 36 36" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m29 4h-25c-1.7 0-3 1.3-3 3v25c0 1.7 1.3 3 3 3h25c1.7 0 3-1.3 3-3v-25c0-1.7-1.3-3-3-3zm-2 28h-23v-23h23z" fill="#ff0000"/><path d="m29 4h-25c-1.7 0-3 1.3-3 3v25c0 1.7 1.3 3 3 3h25c1.7 0 3-1.3 3-3v-25c0-1.7-1.3-3-3-3zm-2 28h-23v-23h23z" fill="#ffffff"/></svg>
Valid SVG: True


# Attempted Batch Inference but did not use for final submission.

In [22]:
# def generate_svg_batch(prompts, max_new_tokens=1024):
#     tokenizer.padding_side = "left"
#     if tokenizer.pad_token is None:
#       tokenizer.pad_token = tokenizer.eos_token

#     # Create the chat structure for the entire batch
#     batch_text = [
#         f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
#         f"<|im_start|>user\n{p}<|im_end|>\n"
#         f"<|im_start|>assistant\n"
#         for p in prompts
#     ]

#     # Tokenize everything at once with padding
#     inputs = tokenizer(text=batch_text, return_tensors="pt", padding=True).to(model.device)
#     input_lengths = inputs.input_ids.shape[1]

#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False, # Stick to greedy for speed and structure
#             repetition_penalty=1.0,
#             eos_token_id=tokenizer.eos_token_id,
#             pad_token_id=tokenizer.pad_token_id # Explicitly pass pad_token_id
#         )

#     # Decode the batch
#     responses = []
#     for i in range(len(prompts)):
#         # Extract only the new tokens for each specific row
#         decoded = tokenizer.decode(output_ids[i][input_lengths:], skip_special_tokens=True)
#         responses.append(extract_svg(decoded))

#     return responses

In [23]:
# # Force-set everything in the global scope
# tokenizer.padding_side = "left"
# tokenizer.pad_token = tokenizer.eos_token

# model.config.pad_token_id = tokenizer.pad_token_id
# model.config.padding_side = "left"

# # If using Unsloth/Peft
# if hasattr(model, "generation_config"):
#     model.generation_config.pad_token_id = tokenizer.pad_token_id
#     model.generation_config.padding_side = "left"

In [24]:
# from tqdm.auto import tqdm
# TEST_PROMPTS_PATH = "/content/drive/MyDrive/test.csv"
# SUBMISSION_PATH = "/content/drive/MyDrive/submission_8192.csv"

# test_df = pd.read_csv(TEST_PROMPTS_PATH)
# BATCH_SIZE = 16  # You can likely push this to 16 or 32 given your 95GB VRAM
# rows = []
# invalid_count = 0
# t0 = time.time()

# # Process in chunks
# for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Generating Batches"):
#     batch_df = test_df.iloc[i : i + BATCH_SIZE]
#     prompts = batch_df["prompt"].tolist()
#     ids = batch_df["id"].tolist()

#     # Generate all SVGs in this batch at once
#     batch_svgs = generate_svg_batch(prompts)


#     for idx, (row_id, svg, prompt) in enumerate(zip(ids, batch_svgs, prompts)):
#         print(svg)
#         if not is_valid_svg(svg):
#             invalid_count += 1
#             # print(f"Fallback triggered for ID: {row_id}")
#             print(f"current count: {invalid_count}. failed svg: {svg}")
#             svg = fallback_svg(prompt)

#         rows.append({"id": row_id, "svg": svg})

# # Save results
# sub_df = pd.DataFrame(rows)
# sub_df.to_csv(SUBMISSION_PATH, index=False)

# print(f"\nDone! Runtime: {(time.time()-t0)/60:.2f} min | Invalids: {invalid_count}")

Generating Batches:   0%|          | 0/63 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



current count: 1. failed svg: 

current count: 2. failed svg: 

current count: 3. failed svg: 

current count: 4. failed svg: 
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#333333" fill-opacity="1"  filling="0" d="M100 183.33 C98.33 183.33 96.67 182.5 95.83 181.25 L18.75 64.17 C17.5 62.5 17.5 60 18.75 58.33 C20 56.67 22.5 56.67 23.75 58.33 L100 134.17 L176.25 58.33 C177.5 57.5 179.17 56.67 180.83 56.67 C182.5 56.67 183.75 57.5 185 58.75 C186.25 60 186.25 61.67 185.83 63.33 L108.75 180.42 C107.92 182.08 106.25 183.33 104.58 183.33 L100 183.33 Z"></path></svg>

current count: 5. failed svg: 

current count: 6. failed svg: 

current count: 7. failed svg: 

current count: 8. failed svg: 

current count: 9. failed svg: 

current count: 10. failed svg: 

current count: 11. failed svg: 

current count: 12. failed svg: 

current count: 13. failed svg: 

current count: 14. failed svg: 

current count: 15. failed svg: 


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



current count: 16. failed svg: 

current count: 17. failed svg: 

current count: 18. failed svg: 

current count: 19. failed svg: 

current count: 20. failed svg: 

current count: 21. failed svg: 

current count: 22. failed svg: 

current count: 23. failed svg: 

current count: 24. failed svg: 
<svg fill="none" height="128" viewBox="0 0 24 24" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m12 22c-0.55 0-1-0.45-1-1v-18c0-0.550.45-1 1-1s1 0.45 1 1v18c0 0.55-0.45 1-1 1zm0-20c-0.55 0-1 0.45-1 1v18c0 0.550.45 1 1 1s1-0.45 1-1v-18c0-0.55-0.45-1-1-1z" stroke="#000000" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"/></svg>

current count: 25. failed svg: 

current count: 26. failed svg: 

current count: 27. failed svg: 

current count: 28. failed svg: 

current count: 29. failed svg: 

current count: 30. failed svg: 


...

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



current count: 162. failed svg: 

current count: 163. failed svg: 

current count: 164. failed svg: 

current count: 165. failed svg: 

current count: 166. failed svg: 

current count: 167. failed svg: 

current count: 168. failed svg: 

current count: 169. failed svg: 

current count: 170. failed svg: 

current count: 171. failed svg: 

current count: 172. failed svg: 

current count: 173. failed svg: 

current count: 174. failed svg: 

current count: 175. failed svg: 

current count: 176. failed svg: 

current count: 177. failed svg: 


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



current count: 178. failed svg: 

current count: 179. failed svg: 

current count: 180. failed svg: 

current count: 181. failed svg: 

current count: 182. failed svg: 

current count: 183. failed svg: 

current count: 184. failed svg: 

current count: 185. failed svg: 

current count: 186. failed svg: 

current count: 187. failed svg: 
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#000000" fill-opacity="1"  filling="0" d="M100 16.67 C146.02 16.67 183.33 53.97 183.33 100 C183.33 146.02 146.02 183.33 100 183.33 C53.97 183.33 16.67 146.02 16.67 100 C16.67 53.97 53.97 16.67 100 16.67 Z M100 33.33 C61.75 33.33 30.55 64.53 30.55 102.78 C30.55 141.03 61.75 172.22 100 172.22 C138.25 172.22 169.45 141.03 169.45 102.78 C169.45 64.53 138.25 33.33 100 33.33 Z M100 125 L100 133.33 L116.67 133.33 L116.67 125 L100 125 Z M100 108.33 L100 116.67 L116.67 116.67 L116.67 108.33 L100 108.33 Z M100 91.67 L100 100 L116.67 100 L116.67 91.67 L100 91.67

KeyboardInterrupt: 

#Batch Inference End

In [29]:
import pandas as pd
import time
from tqdm.auto import tqdm  # .auto automatically chooses between notebook/console UI



# Paths
TEST_PROMPTS_PATH = "/content/drive/MyDrive/test.csv"
SUBMISSION_PATH = "/content/drive/MyDrive/submission_64_64_MSL8192_DST.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

rows = []
invalid_count = 0
t0 = time.time()

# Wrap the dataframe iterator with tqdm
# total=len(test_df) ensures the progress bar knows how far to go
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating SVGs"):
    svg = generate_svg(row["prompt"])
    print(svg)
    if not is_valid_svg(svg):
        invalid_count += 1
        print(f"current count: {invalid_count}. failed svg: {svg}")
        svg = fallback_svg(row["prompt"])

    rows.append({"id": row["id"], "svg": svg})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\nSaved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")

Generating SVGs:   0%|          | 0/1000 [00:00<?, ?it/s]


current count: 1. failed svg: 

current count: 2. failed svg: 

current count: 3. failed svg: 

current count: 4. failed svg: 
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#4D4D4D" fill-opacity="1"  filling="0" d="M98.67 15.67 L101.33 15.67 C103.33 15.67 105 17.33 105 19.33 L105 180.67 C105 182.67 103.33 184.33 101.33 184.33 L98.67 184.33 C96.67 184.33 95 182.67 95 180.67 L95 19.33 C95 17.33 96.67 15.67 98.67 15.67 Z"></path></svg>

current count: 5. failed svg: 

current count: 6. failed svg: 

current count: 7. failed svg: 

current count: 8. failed svg: 
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200px" width="200px"><path fill="#4D4D4D" fill-opacity="1"  filling="0" d="M98.75 16.67 C100.83 16.67 102.5 17.5 103.75 19.17 L140.83 56.25 C142.5 57.5 143.33 59.17 143.33 60.83 C143.33 62.5 142.5 64.17 140.83 65.42 L103.75 102.5 C102.5 103.75 100.83 104.58 98.75 104.58 C96.67 104.58 95 103.75 93.75 102.5 

#If you wish to run inference on an existing model state, plug in model folder to MODEL_PATH variable

In [ ]:
# # SETUP
# from google.colab import drive
# drive.mount('/content/drive')

# import pandas as pd
# import time
# import re
# import xml.etree.ElementTree as ET
# from tqdm.auto import tqdm

# import torch
# from unsloth import FastLanguageModel
# from peft import PeftModel

# MODEL_PATH = "/content/drive/MyDrive/qwen2b_64x64_clean_drop_LR2_8192_weight_decay_0.01_last"
# TEST_PROMPTS_PATH = "/content/drive/MyDrive/test.csv"
# SUBMISSION_PATH = "/content/drive/MyDrive/submission2.csv"

# print("Saved files:", os.listdir(MODEL_PATH))

# for name, module in model.named_modules():
#   if "lora" in name.lower():
#     print(name)
#     break

In [ ]:

# model, tokenizer = FastLanguageModel.from_pretrained(
#   model_name=CONFIG["model_name"],
#   max_seq_length=3072,
#   dtype=None,
#   load_in_4bit=False,
# )

# FastLanguageModel.for_inference(model)

# SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)


# def extract_svg(text):
#     m = SVG_REGEX.search(text)
#     return m.group(0).strip() if m else ""


# def is_valid_svg(svg_text):
#     if not svg_text:
#         return False
#     try:
#         root = ET.fromstring(svg_text)
#         return root.tag.endswith("svg")
#     except ET.ParseError:
#         return False


# def fallback_svg(_prompt):
#     return (
#         '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
#         '<rect x="0" y="0" width="256" height="256" fill="white"/>'
#         '<circle cx="128" cy="128" r="64" fill="black"/>'
#         '</svg>'
#     )


# def generate_svg(prompt, max_new_tokens=1024):
#     chat_text = (
#         "<|im_start|>system\n"
#         f"{SYSTEM_PROMPT}<|im_end|>\n"
#         "<|im_start|>user\n"
#         f"{prompt}<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )
#     inputs = tokenizer(text=[chat_text], return_tensors="pt").to(model.device)
#     # Measure how many tokens were in your prompt
#     input_length = inputs.input_ids.shape[1]
#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=True,
#             temperature=0.1,
#             top_p=0.9,
#             repetition_penalty=1.1,
#             eos_token_id=tokenizer.eos_token_id,
#             pad_token_id=tokenizer.pad_token_id
#         )

#     decoded_new = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True)
#     # decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
#     svg = extract_svg(decoded_new)

#     # if not is_valid_svg(svg):
#     #     print("failed")
#     #     svg = fallback_svg(prompt)

#     return svg


# #test_prompt = "a simple blue bird icon"
# #test_prompt = "The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background."
# test_prompt = "Give me a red square"
# pred_svg = generate_svg(test_prompt)
# print(pred_svg)
# print("Valid SVG:", is_valid_svg(pred_svg))




# model = PeftModel.from_pretrained(model, MODEL_PATH)
# FastLanguageModel.for_inference(model)

# print("Model type:", type(model))
# print("First parameter device:", next(model.parameters()).device)
# print("Model + LoRA loaded")


# test_df = pd.read_csv(TEST_PROMPTS_PATH)


# rows = []
# invalid_count = 0
# t0 = time.time()

# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating SVGs"):
#     svg = generate_svg(row["prompt"])
#     print(svg)
#     if not is_valid_svg(svg):
#         invalid_count += 1
#         print(f"current count: {invalid_count}. failed svg: {svg}")
#         svg = fallback_svg(row["prompt"])

#     rows.append({"id": row["id"], "svg": svg})


# sub_df = pd.DataFrame(rows)
# sub_df.to_csv(SUBMISSION_PATH, index=False)

# elapsed_min = (time.time() - t0) / 60

# print("\nDONE")
# print(f"Saved: {SUBMISSION_PATH}")
# print(f"Rows: {len(sub_df)}")
# print(f"Invalid/fallback count: {invalid_count}")
# print(f"Runtime (minutes): {elapsed_min:.2f}")

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.